In [2]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import torch
from torch.utils.data import TensorDataset, DataLoader, Subset
import torch.nn as nn
import torch.nn.functional as F
from itertools import product
from sklearn.metrics import roc_auc_score
from sklearn.model_selection import StratifiedKFold
import pickle

In [3]:
#Leer pkl con el tensor
file_path = 'stamps_tensor.pkl'
with open(file_path, 'rb') as f:
        dataset = pickle.load(f)

In [11]:
df_stamps = pd.read_pickle("alerce_stamps/stamps_onecandid.pkl")
# index es oid
df_stamps = df_stamps.set_index("oid")

In [12]:
df_indexed = df_stamps.reset_index(drop=False) 
df_indexed['orig_idx'] = df_indexed.index

In [13]:
# extraemos las dimensiones de los stamps
TARGET_H, TARGET_W = 63, 63
N = len(df_stamps)
print(f"Dimensiones de los stamps: {TARGET_H}x{TARGET_W}, Número de stamps: {N}")

Dimensiones de los stamps: 63x63, Número de stamps: 19730


In [14]:
# K folds estratisficados
splits = pd.read_csv('folds/stratified_folds_stamps.csv') 

In [15]:
df_merged = df_indexed.merge(splits, on='oid', how='inner')
assert len(df_merged) == N, f"Mismatch: df_merged has {len(df_merged)} rows, expected {N}"

In [16]:
# GPU
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"Using device: {device}")

Using device: cuda


In [17]:
loader = DataLoader(
    dataset,
    batch_size=16,
    shuffle=True,
    num_workers=2,
    pin_memory=True,
    persistent_workers=True,
    prefetch_factor=2
)

In [18]:
class STAMPCNN(nn.Module):
    """
    CNN para procesar un stamp con 3 canales:
    - channel 0: difference
    - channel 1: science
    - channel 2: template
    Entrada: (batch_size, 3, 63, 63)
    Salida:  (batch_size, out_features)
    """
    def __init__(self, out_features: int = 64, dropout: float = 0.3):
        super().__init__()
        # Convoluciones + batchnorm
        self.conv1 = nn.Conv2d(3, 16, kernel_size=5, padding=2)
        self.bn1   = nn.BatchNorm2d(16)
        self.conv2 = nn.Conv2d(16, 32, kernel_size=3, padding=1)
        self.bn2   = nn.BatchNorm2d(32)

        # Poolings
        self.pool = nn.MaxPool2d(2, 2)

        # Aplanamiento y FCs perezosos (LazyLinear infiere in_features)
        self.flatten = nn.Flatten()
        self.fc1     = nn.LazyLinear(128)     # infiere automáticamente in_features
        self.dropout = nn.Dropout(dropout)
        self.fc2     = nn.Linear(128, out_features)

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        # Bloque 1
        x = F.relu(self.bn1(self.conv1(x)))
        x = self.pool(x)
        # Bloque 2
        x = F.relu(self.bn2(self.conv2(x)))
        x = self.pool(x)
        # Aplanar y FCs
        x = self.flatten(x)
        x = F.relu(self.fc1(x))
        x = self.dropout(x)
        return self.fc2(x)

In [19]:
# numero de epochs (bajo para gridsearch)
num_epochs = 3

In [20]:
"""
param_grid = {
    'lr':        [1e-3, 5e-4, 1e-4],
    'batch_size':[8, 16, 32],
    'dropout':   [0.0, 0.3, 0.5],
}
"""
param_grid = {
    'lr':        [1e-3],
    'batch_size':[16, 32],
    'dropout':   [0.3, 0.5],
}

In [21]:
best_auc    = -float('inf')
best_params = None

In [22]:
def get_fold_indices(df, fold):
    train_col = f'train_fold{fold}'
    val_col   = f'val_fold{fold}'
    train_idx = df.loc[df[train_col]==1, 'orig_idx'].tolist()
    val_idx   = df.loc[df[val_col]==1,   'orig_idx'].tolist()
    return train_idx, val_idx

In [23]:
for fold in range(1,6):
    _, val_idx = get_fold_indices(df_merged, fold)
    lbls = df_merged.loc[df_merged['orig_idx'].isin(val_idx), 'label']
    n_pos = (lbls=='good').sum()
    n_neg = (lbls=='bad').sum()
    print(f"Fold {fold}: +={n_pos}, -={n_neg}")

for lr, batch_size, dropout in product(
        param_grid['lr'],
        param_grid['batch_size'],
        param_grid['dropout']
    ):

    cv_aucs = []

    for fold in range(1, 6):
        # 1) Índices de este fold
        train_idx, val_idx = get_fold_indices(df_merged, fold)

        # 2) DataLoaders
        tr_loader = DataLoader(
            Subset(dataset, train_idx),
            batch_size=batch_size, shuffle=True,
            pin_memory=True, num_workers=2, persistent_workers=True
        )
        vl_loader = DataLoader(
            Subset(dataset, val_idx),
            batch_size=batch_size, shuffle=False,
            pin_memory=True, num_workers=2, persistent_workers=True
        )

        # 3) Modelo y optimizador
        model = STAMPCNN(out_features=1, dropout=dropout).to(device)
        optim = torch.optim.Adam(model.parameters(), lr=lr)
        crit  = nn.BCEWithLogitsLoss()

        # 4) Entrenamiento
        for epoch in range(num_epochs):
            model.train()
            for data, labels in tr_loader:
                x = data.to(device, non_blocking=True)   # ahora (batch,3,63,63)
                y = labels.to(device, non_blocking=True).float()
                optim.zero_grad()
                logits = model(x).squeeze(1)              # (batch,)
                loss   = crit(logits, y)
                loss.backward()
                optim.step()

        # 5) Validación
        model.eval()
        all_logits = []
        all_labels = []
        with torch.no_grad():
            for data, labels in vl_loader:
                x = data.to(device, non_blocking=True)
                logits = model(x).squeeze(1).cpu()      # (batch,)
                all_logits.append(logits)
                all_labels.append(labels)

        # Concatenar todo el fold
        logits = torch.cat(all_logits)               # tensor (M,)
        labels = torch.cat(all_labels)               # tensor (M,)

        # Convertir a probabilidades
        probs = torch.sigmoid(logits)

        # Filtrado de NaN (en labels o en probs)
        mask_valid = (~torch.isnan(labels)) & (~torch.isnan(probs))
        clean_labels = labels[mask_valid].numpy()
        clean_probs  = probs[mask_valid].numpy()

        # Comprobar que hay al menos una clase de cada tipo
        if len(clean_labels) >= 2 and len(np.unique(clean_labels)) == 2:
            auc = roc_auc_score(clean_labels, clean_probs)
        else:
            # Si no hay suficiente variedad, salta este fold
            auc = float('nan')
        cv_aucs.append(auc)
        print(f"Fold {fold}, AUC: {auc:.4f} (lr={lr}, batch={batch_size}, drop={dropout})")

    # 6) Promedio de AUC y actualización de best
    valid_aucs = [a for a in cv_aucs if not np.isnan(a)]
    if valid_aucs:
        mean_auc = sum(valid_aucs) / len(valid_aucs)
    else:
        mean_auc = float('nan')
    print(f"lr={lr}, batch={batch_size}, drop={dropout} → AUC={mean_auc:.4f}")
    if not np.isnan(mean_auc) and mean_auc > best_auc:
        best_auc    = mean_auc
        best_params = (lr, batch_size, dropout)

print("Mejores params STAMPCNN:", best_params, "AUC:", best_auc)

Fold 1: +=2163, -=994
Fold 2: +=2163, -=994
Fold 3: +=2163, -=994
Fold 4: +=2163, -=994
Fold 5: +=2162, -=994
Fold 1, AUC: 0.7603 (lr=0.001, batch=16, drop=0.3)
Fold 2, AUC: 0.7837 (lr=0.001, batch=16, drop=0.3)
Fold 3, AUC: 0.7992 (lr=0.001, batch=16, drop=0.3)


KeyboardInterrupt: 